In [1]:
import pandas as pd
import numpy as np
import pickle
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report

In [2]:
X_train = pd.read_csv('../data/X_train.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
X_test  = pd.read_csv('../data/X_test.csv')
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

In [5]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos:.2f}")

Scale pos weight: 1.75


In [6]:
param_dist = {
    'max_depth':     [5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators':  [100, 200],
    'subsample':     [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
}

In [14]:
xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss', device='cuda'
)

In [13]:
search = RandomizedSearchCV(
    xgb_base, param_dist,
    n_iter=100,        # try 10 random combos (fast enough)
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

In [19]:
search.fit(X_train, y_train)

print(f"Best params: {search.best_params_}")
print(f"Best F1: {search.best_score_:.4f}")

pickle.dump(search.best_estimator_,
            open('../results/xgboost_tuned_model.pkl', 'wb'))
print("Tuned model saved.")

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.2, 'colsample_bytree': 0.7}
Best F1: 0.9997
Tuned model saved.


In [18]:
y_pred_tuned = search.best_estimator_.predict(X_test)

print(f"F1 Score (tuned): {f1_score(y_test, y_pred_tuned):.4f}")
print(classification_report(y_test, y_pred_tuned))

c:\Users\Maneshna\miniconda3\envs\torch_env\lib\site-packages\xgboost\core.py:751: UserWarning: [01:53:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


F1 Score (tuned): 0.9999
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     87937
           1       1.00      1.00      1.00     50345

    accuracy                           1.00    138282
   macro avg       1.00      1.00      1.00    138282
weighted avg       1.00      1.00      1.00    138282

